In [5]:
pip install pandas statsmodels openpyxl

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Usuario\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [14]:
with open("base.csv", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(repr(f.readline()))

'\ufeffcliente_etica,canal,cohorte_año,antiguedad_meses,venta_no_etica_historica,log_venta_no_etica,productos_no_etica_historicos\n'
'"1,VENTAS AL PERSONAL,2022,47,761634.0000,""5,881746893303946"",115"\n'
'"1,INSTITUCIONALES,2022,48,73559.0000,""4,86664172056604"",16"\n'
'"0,CLIENTES VARIOS,2022,45,1468.0000,""3,1670217957902564"",1"\n'
'"1,LICITACIONES,2022,47,122704.0000,""5,0888622597780815"",10"\n'


In [11]:
import pandas as pd

df = pd.read_csv("base.csv", sep=",")

In [30]:
import pandas as pd
from io import StringIO

with open("base2.csv", "r", encoding="utf-8-sig") as archivo:
    lineas = archivo.readlines()

# El encabezado está correctamente formado
encabezado = lineas[0]

filas_corregidas = []

for linea in lineas[1:]:
    linea = linea.strip()

    # Quitar las comillas que envuelven toda la fila
    if linea.startswith('"') and linea.endswith('"'):
        linea = linea[1:-1]

    # Convertir comillas duplicadas "" en comillas normales "
    linea = linea.replace('""', '"')

    filas_corregidas.append(linea)

contenido_corregido = encabezado + "\n".join(filas_corregidas)

df = pd.read_csv(
    StringIO(contenido_corregido),
    sep=",",
    decimal=","
)

print(df.head())
print(df.columns)
df.info()

   cliente_etica               canal  cohorte_año  antiguedad_meses  \
0              1  VENTAS AL PERSONAL         2022                47   
1              1     INSTITUCIONALES         2022                48   
2              0     CLIENTES VARIOS         2022                45   
3              1        LICITACIONES         2022                47   
4              1            CLINICAS         2024                29   

  venta_historica  log_venta_historica  productos_no_etica_historicos  \
0     797324.0461             5.901635                            115   
1      99151.3300             4.996303                             16   
2       1468.7200             3.167235                              1   
3     123044.9600             5.090067                             10   
4       2929.8100             3.466988                              3   

   lineas_historicas frecuencia_historica  meses_activos_historicos  \
0                  6             114.1364                      

In [31]:
print(df.head())

   cliente_etica               canal  cohorte_año  antiguedad_meses  \
0              1  VENTAS AL PERSONAL         2022                47   
1              1     INSTITUCIONALES         2022                48   
2              0     CLIENTES VARIOS         2022                45   
3              1        LICITACIONES         2022                47   
4              1            CLINICAS         2024                29   

  venta_historica  log_venta_historica  productos_no_etica_historicos  \
0     797324.0461             5.901635                            115   
1      99151.3300             4.996303                             16   
2       1468.7200             3.167235                              1   
3     123044.9600             5.090067                             10   
4       2929.8100             3.466988                              3   

   lineas_historicas frecuencia_historica  meses_activos_historicos  \
0                  6             114.1364                      

In [32]:
import statsmodels.formula.api as smf

In [35]:
modelo_productos = smf.ols(
    formula="""
        productos_no_etica_historicos
        ~ cliente_etica
        + antiguedad_meses
        + log_venta_historica
        + C(canal)
        + C(cohorte_año)
    """,
    data=df
).fit(cov_type="HC3")

In [36]:
modelo_lineas = smf.ols(
    formula="""
        lineas_historicas
        ~ cliente_etica
        + antiguedad_meses
        + log_venta_historica
        + C(canal)
        + C(cohorte_año)
    """,
    data=df
).fit(cov_type="HC3")

In [40]:
import pandas as pd

columnas_numericas = [
    "cliente_etica",
    "cohorte_año",
    "antiguedad_meses",
    "venta_historica",
    "log_venta_historica",
    "productos_no_etica_historicos",
    "lineas_historicas",
    "frecuencia_historica",
    "meses_activos_historicos",
    "venta_mensual_promedio_historica"
]

for columna in columnas_numericas:
    df[columna] = (
        df[columna]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.strip()
    )

    df[columna] = pd.to_numeric(
        df[columna],
        errors="coerce"
    )

In [41]:
modelo_frecuencia = smf.ols(
    formula="""
        frecuencia_historica
        ~ cliente_etica
        + antiguedad_meses
        + log_venta_historica
        + C(canal)
        + C(cohorte_año)
    """,
    data=df
).fit(cov_type="HC3")

In [43]:
modelo_meses = smf.ols(
    formula="""
        meses_activos_historicos
        ~ cliente_etica
        + antiguedad_meses
        + log_venta_historica
        + C(canal)
        + C(cohorte_año)
    """,
    data=df
).fit(cov_type="HC3")

In [44]:
modelo_venta = smf.ols(
    formula="""
        venta_mensual_promedio_historica
        ~ cliente_etica
        + antiguedad_meses
        + log_venta_historica
        + C(canal)
        + C(cohorte_año)
    """,
    data=df
).fit(cov_type="HC3")

In [45]:
print(modelo_productos.summary())
print(modelo_lineas.summary())
print(modelo_frecuencia.summary())
print(modelo_meses.summary())
print(modelo_venta.summary())

                                  OLS Regression Results                                 
Dep. Variable:     productos_no_etica_historicos   R-squared:                       0.666
Model:                                       OLS   Adj. R-squared:                  0.665
Method:                            Least Squares   F-statistic:                     422.2
Date:                           Mon, 20 Jul 2026   Prob (F-statistic):               0.00
Time:                                   20:22:19   Log-Likelihood:                -29188.
No. Observations:                           7695   AIC:                         5.842e+04
Df Residuals:                               7671   BIC:                         5.859e+04
Df Model:                                     23                                         
Covariance Type:                             HC3                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
-

In [46]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Variables explicativas
X = df[
    [
        "cliente_etica",
        "antiguedad_meses",
        "log_venta_historica",
    ]
].copy()

# Variables categóricas
X = pd.get_dummies(
    pd.concat([X, df[["canal", "cohorte_año"]]], axis=1),
    columns=["canal", "cohorte_año"],
    drop_first=True
)

# Asegurar que todo sea numérico
X = X.astype(float)

vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif = vif.sort_values("VIF", ascending=False)

print(vif)

                    Variable         VIF
9            canal_FARMACIAS  237.139733
1           antiguedad_meses  235.073499
2        log_venta_historica   47.735591
5      canal_CLIENTES VARIOS   29.662146
21          cohorte_año_2025   20.203710
10     canal_INSTITUCIONALES   17.046502
0              cliente_etica   12.908479
20          cohorte_año_2024    9.791329
4             canal_CATALOGO    7.824231
6             canal_CLINICAS    7.325678
22          cohorte_año_2026    6.713517
12          canal_MAYORISTAS    5.451958
7          canal_CORPORATIVO    4.321463
11        canal_LICITACIONES    3.887835
18        canal_VETERINARIAS    3.880696
13             canal_MEDICOS    3.248467
8       canal_DISTRIBUIDORES    3.237703
19          cohorte_año_2023    3.198360
17    canal_VENTAS MOSTRADOR    2.467891
15       canal_SUPERMERCADOS    2.322924
14     canal_MINI MAYORISTAS    2.167049
3              canal_CADENAS    2.080475
16  canal_VENTAS AL PERSONAL    1.044085


In [47]:
import matplotlib.pyplot as plt

variables = [
    "venta_historica",
    "venta_no_etica_historica",
    "productos_no_etica_historicos",
    "productos_historicos",
    "lineas_historicas",
    "facturas_historicas",
    "frecuencia_historica",
    "meses_activos_historicos",
    "venta_mensual_promedio_historica",
    "antiguedad_meses",
]

corr = df[variables].corr(numeric_only=True)

print(corr.round(3))

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.tight_layout()
plt.show()

KeyError: "['venta_no_etica_historica', 'productos_historicos', 'facturas_historicas'] not in index"